# LangChain Prompting – Setup

**Goal:** Run simple prompts using the Groq chat model.

### 1. Install packages
Run this cell once in your notebook. If you install packages while Jupyter is open, restart the kernel if imports fail.

In [ ]:
%pip install -q -U langchain langchain-core langchain-groq python-dotenv

### 2. Configure your Groq API key with a `.env` file

For local development, use `python-dotenv` to load the API key from a `.env` file. This keeps credentials out of notebook code and makes the setup repeatable.

1. In the same project folder as these notebooks, create a file named `.env`.
2. Add this line, replacing the placeholder with your own Groq API key:

   `GROQ_API_KEY=your_actual_groq_api_key`

3. Keep `.env` private. The included `.gitignore` excludes it from Git.
4. Run the next setup cell before initializing the model.

Do not commit, upload, or share your real `.env` file. The included `.env.example` is only a template and contains no real credential.


In [1]:
# Load credentials from the local .env file.
import os
from dotenv import load_dotenv

load_dotenv()  # Reads .env from the current project directory (or parent directories).

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
# if not GROQ_API_KEY:
#     raise ValueError(
#         "GROQ_API_KEY was not found. Create a .env file beside the notebooks "
#         "with: GROQ_API_KEY=your_actual_groq_api_key"
    # )

print("Groq API key loaded successfully.")


Groq API key loaded successfully.


### 3. Create the chat model

`ChatGroq` connects LangChain to a Groq-hosted chat model. Model names can change, so use a model currently available in your Groq account.

In [2]:
from langchain_groq import ChatGroq

# Keep the model name configurable because provider model availability can change.
MODEL_NAME = "openai/gpt-oss-120b"

llm = ChatGroq(
    model=MODEL_NAME,
    temperature=0,
    api_key=GROQ_API_KEY
)
print("Groq chat model initialized.")


Groq chat model initialized.


> If Groq reports that the model name is unavailable or decommissioned, replace `model` with a currently supported chat model from your Groq Console.

> **Security and setup:** The original hardcoded API keys have been removed. Run the secure key-entry cell and model setup cell before running model examples. Use a newly generated key; revoke any key that was previously exposed. The model name shown is the one used in the source notebook; change `MODEL_NAME` if it is no longer available in your Groq account.

# Day 3 – ChatPromptTemplate

## Learning objectives
- Understand chat messages and their roles.
- Use system, human, and AI messages.
- Build reusable chat prompts.
- Insert chat history with MessagesPlaceholder.

## 1. Chat messages

- **System:** Sets the assistant's behaviour.
- **Human:** Contains the user's message.
- **AI:** Represents an assistant message.

In [3]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful Python teacher. Use simple English."),
    ("human", "Explain {topic} with one example.")
])

messages = chat_prompt.format_messages(topic="Python dictionaries")
for message in messages:
    print(type(message).__name__, ":", message.content)

SystemMessage : You are a helpful Python teacher. Use simple English.
HumanMessage : Explain Python dictionaries with one example.


## 2. Send the chat prompt to Groq

In [4]:
chain = chat_prompt | llm
response = chain.invoke({"topic": "Python functions"})
print(response.content)

**What is a function?**  

A function is a reusable block of code that does a specific job.  
You write the code once, give the function a name, and then you can “call” (run) it many times whenever you need it.  

---

## Basic parts of a Python function  

```python
def function_name(parameters):
    """optional docstring – explains what the function does"""
    # body – one or more statements that do the work
    return result          # optional, sends a value back to the caller
```

| Part | What it means |
|------|---------------|
| `def` | Keyword that tells Python we are defining a function. |
| `function_name` | Any name you choose (follow normal variable‑naming rules). |
| `parameters` | Values the function needs to work with; they go inside the parentheses. If the function needs nothing, leave the parentheses empty: `def hello():`. |
| `:` | Ends the header line and starts the indented block (the body). |
| `return` | Sends a value back to the place where the function was cal

## 3. Add more than one message

In [5]:
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an IT career guidance assistant."),
    ("human", "I am learning {skill}."),
    ("ai", "That is a useful skill to learn."),
    ("human", "Suggest a beginner project using {skill}.")
])

response = (chat_prompt | llm).invoke({
    "skill": "Python"
})
print(response.content)

Sure! Here are a few beginner‑friendly Python projects that let you practice core concepts (variables, control flow, functions, file I/O, and a bit of libraries) while building something useful or fun. Pick the one that excites you most, and feel free to add your own twists.

---

## 1. **Number Guessing Game**
**What it does:** The computer picks a random number (e.g., 1‑100). The player tries to guess it, and the program tells them “too high”, “too low”, or “correct!” after each guess.

**Key concepts practiced**
- `random` module (`random.randint`)
- Loops (`while`)
- Conditional statements (`if/elif/else`)
- Input handling (`input()`)
- Basic functions (optional: wrap the game logic in a function)

**Stretch goals**
- Limit the number of attempts and display a score.
- Add a “play again?” loop.
- Store high scores in a text file.

---

## 2. **Simple To‑Do List (CLI)**
**What it does:** Users can add, view, mark as done, and delete tasks from a list that persists between runs.

**K

## 4. MessagesPlaceholder – insert chat history

A placeholder lets you insert a list of previous messages into a chat prompt.

In [6]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use the conversation history when relevant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

history = [
    HumanMessage(content="I am learning Python."),
    AIMessage(content="Great! We can practise Python basics.")
]

messages = chat_prompt.format_messages(
    history=history,
    question="Which topic did I say I was learning?"
)

for message in messages:
    print(type(message).__name__, ":", message.content)

response = llm.invoke(messages)
print("\nANSWER:")
print(response.content)

SystemMessage : You are a helpful assistant. Use the conversation history when relevant.
HumanMessage : I am learning Python.
AIMessage : Great! We can practise Python basics.
HumanMessage : Which topic did I say I was learning?

ANSWER:
You mentioned that you’re learning **Python**.


## 5. Simple reusable chat function

In [7]:
def ask_tutor(topic, level="beginner"):
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a patient programming teacher."),
        ("human", "Teach me {topic}. My level is {level}. Give a small code example.")
    ])
    return (prompt | llm).invoke({
        "topic": topic,
        "level": level
    }).content

print(ask_tutor("Python exception handling"))

## Python Exception Handling – A Gentle Introduction  

When you write a program, things don’t always go exactly as you expect.  
For example, you might try to open a file that isn’t there, or you might ask the user to type a number but they type letters instead.  
If you don’t tell Python how to deal with those “surprises”, the program will stop with a **traceback** (an error message) and exit.

**Exception handling** is the way we tell Python:

> “If something goes wrong, *catch* the problem and decide what to do instead of crashing.”

---

### The basic building blocks

| Keyword | What it does | Typical use |
|---------|--------------|-------------|
| `try`   | Put the code that *might* raise an error here. | Anything that could fail (file I/O, converting input, network calls, …) |
| `except`| Catch a specific error (or any error) and run alternative code. | Show a friendly message, use a default value, retry, … |
| `else`  | Code that runs **only if** the `try` block succeeded (no

## Student practice
1. Create a chat prompt with a system message and two human messages.
2. Ask the model to explain SQL to a beginner.
3. Add two previous messages using MessagesPlaceholder.
4. Change the system instruction and compare the answers.

**Remember:** ChatPromptTemplate creates a sequence of messages. It does not itself remember earlier conversations; you must supply history or use a memory/history mechanism.

## 8. Message roles: what each role contributes

- **System:** high-level behaviour and rules for the assistant.
- **Human:** the current user request or supplied input.
- **AI:** an assistant message, often used to represent earlier assistant turns or examples.
- **Tool:** a result returned by a tool in tool-enabled workflows (not the same as an ordinary AI message).

**Why it is useful:** Separating roles helps keep persistent behaviour, user input, and conversation history organized. Exact role support and tool behaviour depend on the model/provider.

In [8]:
from langchain_core.prompts import ChatPromptTemplate

role_demo = ChatPromptTemplate.from_messages([
    ("system", "You are a careful SQL tutor. Explain safely and clearly."),
    ("human", "Explain what a primary key is."),
    ("ai", "A primary key uniquely identifies each row in a table."),
    ("human", "Give one practical example.")
])

for message in role_demo.format_messages():
    print(type(message).__name__, "->", message.content)

SystemMessage -> You are a careful SQL tutor. Explain safely and clearly.
HumanMessage -> Explain what a primary key is.
AIMessage -> A primary key uniquely identifies each row in a table.
HumanMessage -> Give one practical example.


## 9. Conversation history is context, not permanent memory

`MessagesPlaceholder` inserts messages supplied by your application. It does not automatically save history between program runs. Your application must store, retrieve, and pass the messages.

**Why it is useful:** A chatbot can refer to recent turns without asking the user to repeat information. Keep only relevant history to control token use and avoid passing unnecessary personal data.

In [9]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

history_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise learning assistant."),
    MessagesPlaceholder("chat_history", optional=True),
    ("human", "{question}")
])

# No history is also valid because the placeholder is optional.
print(history_prompt.format_messages(question="What is an API?"))

# Supply history when available.
history_messages = [
    HumanMessage(content="I am studying AWS."),
    AIMessage(content="We can practise AWS fundamentals.")
]
print(history_prompt.format_messages(
    chat_history=history_messages,
    question="Suggest a small project."
))

[SystemMessage(content='You are a concise learning assistant.', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is an API?', additional_kwargs={}, response_metadata={})]
[SystemMessage(content='You are a concise learning assistant.', additional_kwargs={}, response_metadata={}), HumanMessage(content='I am studying AWS.', additional_kwargs={}, response_metadata={}), AIMessage(content='We can practise AWS fundamentals.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Suggest a small project.', additional_kwargs={}, response_metadata={})]


## 10. Limit the history window

For a long-running chatbot, passing every message forever can increase cost and distract the model. A simple application-level approach is to retain only the most recent turns. More advanced applications can summarize older conversation history.

In [10]:
# Example only: retain the latest four message objects.
recent_history = history_messages[-4:]
print("Messages retained:", len(recent_history))

Messages retained: 2


## 11. Build a reusable chat function with input checks

A function can package prompt construction and model invocation. Input checks prevent empty topics or unsupported levels from being sent accidentally.

In [11]:
def ask_learning_assistant(topic, level="beginner"):
    allowed_levels = {"beginner", "intermediate", "advanced"}
    if not isinstance(topic, str) or not topic.strip():
        raise ValueError("topic must be a non-empty string")
    if level not in allowed_levels:
        raise ValueError(f"level must be one of {sorted(allowed_levels)}")

    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an accurate IT instructor. Do not invent technical facts."),
        ("human", "Teach {topic} to an {level} learner. Give a definition, use case, and example.")
    ])
    return (prompt | llm).invoke({"topic": topic, "level": level}).content

print(ask_learning_assistant("REST API", "beginner"))

## 1. What is a REST API?

**REST** stands for **Representational State Transfer**.  
A **REST API** (sometimes called a *RESTful web service*) is a way for two computer systems to talk to each other over the Web using the same conventions that browsers use to fetch pages.

Key points that make an API “REST‑ful”:

| Principle | What it means for the API |
|-----------|---------------------------|
| **Resources** | Everything you want to work with (users, orders, photos, etc.) is treated as a *resource* that has its own URL (e.g., `https://api.example.com/users`). |
| **Uniform Interface** | The same set of HTTP verbs (methods) are used for all resources: <br>• `GET` – retrieve data <br>• `POST` – create a new resource <br>• `PUT` / `PATCH` – update an existing resource <br>• `DELETE` – remove a resource |
| **Stateless** | Each request contains all the information the server needs to understand it. The server does **not** store session data between requests (authentication tokens are s

## Day 3 completion checklist

- [ ] I can explain the concept without reading the notebook.
- [ ] I understand why and when it is useful.
- [ ] I can modify the example for a different IT topic.
- [ ] I can identify at least one limitation or failure case.
- [ ] I completed the practice task.

**Recommended workflow:** Read the explanation → run the example → change the input → compare outputs → explain the result in your own words.